In [ ]:
import os, json, math, pickle, random, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device, 'Torch:', torch.__version__)

In [ ]:

DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TOKENIZER_PATH = "/kaggle/input/datasets/saiveerendra2151/tokenizer-results/project_bpe_16k.pkl"
KERNEL_CHECKPOINT_PATH = "/kaggle/input/datasets/saiveerendra2151/kernel-mha/kernel_final.pt"
OUTPUT_DIR = Path("/kaggle/working/phase3_kernel_rr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN = 1024
MAX_TARGET_LEN = 512
MAX_CONTEXT_SENT_LEN = 128
NUM_NEIGHBORS = 5
PAD_ID, SEP_ID = 0, 3

# Match the pretrained kernel checkpoint configuration in kernel-phase2.ipynb
VOCAB_SIZE = 8000
MODEL_DIM, MODEL_LAYERS, MODEL_FF = 512, 12, 4096
N_HEADS, N_KV_HEADS = 8, 8
FEAT_DIM, ATTN_CHUNK = 32, 128

# LoRA values reflect the user's Phase 3 RR and Kernel experiments;
# edit only to match the actual run you are reproducing.
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0.05
LORA_PROJECTIONS = ("q_proj", "k_proj", "v_proj", "out_proj")
BATCH_SIZE, EPOCHS, PATIENCE = 16, 30, 4

In [ ]:
from huggingface_hub import hf_hub_download

def dataset_file(name):
    return hf_hub_download(repo_id=DATASET_REPO, filename=name, repo_type="dataset")

names = {
    'cl_train':'Phase_3_RR/Train/CL_train.json',
    'it_train':'Phase_3_RR/Train/IT_train.json',
    'cl_val':'Phase_3_RR/Train/CL_dev.json',
    'it_val':'Phase_3_RR/Train/IT_dev.json',
    'cl_test':'Phase_3_RR/Test/CL_test.json',
    'it_test':'Phase_3_RR/Test/IT_test.json',
    'labels':'Phase_3_RR/Train/label_vocab.json',
}
raw = {}
for key, filename in names.items():
    with open(dataset_file(filename),encoding='utf-8') as f:
        raw[key] = json.load(f)
print({k:len(v) for k,v in raw.items()})

In [ ]:
class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [ ]:
assert Path(TOKENIZER_PATH).exists(), f"Tokenizer missing: {TOKENIZER_PATH}"
with open(TOKENIZER_PATH,'rb') as f:
    tokenizer = pickle.load(f)  # only load your own trusted tokenizer pickle
converter = ToTokensConverter(tokenizer.itos)
print('BPE size:',len(tokenizer.itos))
assert len(tokenizer.itos) <= VOCAB_SIZE, 'Tokenizer vocab exceeds pretrained model embedding size'
labels_data = {str(k):int(v) for k,v in raw['labels'].items()}
NUM_CLASSES = len(labels_data)
assert sorted(labels_data.values()) == list(range(NUM_CLASSES)), labels_data
id2label = {v:k for k,v in labels_data.items()}
print('Labels:',NUM_CLASSES,labels_data)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def chunked_gqa_linear_attn(q, k, v, chunk=128, eps=1e-6, padding_mask = None):
    """
    q: (B, Hq, T, F)   k: (B, Hkv, T, F)   v: (B, Hkv, T, D)   all float32, features >= 0
    Replaces the kv / cumsum / einsum block. Returns out (B, Hq, T, D) and final state (S, z).
    """
    B, Hq, T, Fd = q.shape
    Hkv, D = k.shape[1], v.shape[-1]
    R = Hq // Hkv
    if padding_mask is not None:
        valid = (~padding_mask.bool())[:, None, :, None]
        k = k * valid.to(k.dtype)
        v = v * valid.to(v.dtype)
    q = q.view(B, Hkv, R, T, Fd)                      # query head h uses KV head h // R

    S = q.new_zeros(B, Hkv, Fd, D)                    # running sum of phi(k)^T v
    z = q.new_zeros(B, Hkv, Fd)                       # running sum of phi(k)
    tri = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))

    outs = []
    for s in range(0, T, chunk):
        qc = q[:, :, :, s:s + chunk]                  # (B,Hkv,R,c,F)
        kc = k[:, :, s:s + chunk]                     # (B,Hkv,c,F)
        vc = v[:, :, s:s + chunk]                     # (B,Hkv,c,D)
        c = qc.shape[3]

        a = (qc @ kc.unsqueeze(2).transpose(-1, -2)).masked_fill(~tri[:c, :c], 0.0)  # inside chunk
        num = qc @ S.unsqueeze(2) + a @ vc.unsqueeze(2)                             # past + current
        den = (qc * z[:, :, None, None, :]).sum(-1, keepdim=True) + a.sum(-1, keepdim=True)
        outs.append(num / (den + eps))

        S = S + kc.transpose(-1, -2) @ vc             # update state after the chunk
        z = z + kc.sum(dim=2)

    return torch.cat(outs, dim=3).reshape(B, Hq, T, D), (S, z)


class LinearAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.0, feat_dim=32, chunk=128):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.chunk = chunk
        self.feat_dim = feat_dim
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        # kernel: phi(x) = (gamma * norm(x) + beta)^2  (gamma, beta in the LayerNorm)
        self.q_kernel = nn.Linear(self.d_head, feat_dim, bias=True)
        self.k_kernel = nn.Linear(self.d_head, feat_dim, bias=True)
        self.q_norm = nn.LayerNorm(feat_dim)
        self.k_norm = nn.LayerNorm(feat_dim)

        self.resid_dropout = nn.Dropout(dropout)

    def _project(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        q_phi = self.q_norm(self.q_kernel(q)).float() ** 2     # (B,n_heads,T,F)
        k_phi = self.k_norm(self.k_kernel(k)).float() ** 2     # (B,n_kv,T,F) 
        return q_phi, k_phi, v.float()

    def forward(self, x, padding_mask=None, return_state=False):
        B, T, C = x.shape
        q_phi, k_phi, v = self._project(x)
        with torch.autocast(device_type="cuda", enabled=False):
            out, state = chunked_gqa_linear_attn(q_phi, k_phi, v, self.chunk, padding_mask=padding_mask)
        out = F.rms_norm(out.to(x.dtype), (self.d_head,))
        out = self.out_proj(out.transpose(1, 2).reshape(B, T, C))
        out = self.resid_dropout(out)
        return (out, state) if return_state else out

    @torch.no_grad()
    def step(self, x_t, S, z, eps=1e-6):
        """One-token decoding. x_t: (B,1,C), S: (B,Hkv,F,D), z: (B,Hkv,F), both float32."""
        B = x_t.shape[0]
        q_phi, k_phi, v = self._project(x_t)
        S = S + k_phi.transpose(-1, -2) @ v
        z = z + k_phi.squeeze(2)

        qg = q_phi.reshape(
            B, self.n_kv_heads, self.n_rep, self.feat_dim
        )

        numerator = torch.einsum("bhrf,bhfd->bhrd", qg, S)
        denominator = (qg * z.unsqueeze(2)).sum(-1, keepdim=True)

        out = numerator / (denominator + eps)
        out = F.rms_norm(out.to(x_t.dtype), (self.d_head,))
        out = out.reshape(B, 1, self.n_heads * self.d_head)
        out = self.out_proj(out)

        return out, S, z

In [ ]:
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    feat_dim: int = 32                 # kernel feature dimension F
    chunk: int = 128 


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)



class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = LinearAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads,
                                       cfg.dropout, cfg.feat_dim, cfg.chunk)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, padding_mask=None, return_state=False):
            if return_state:
                a, state = self.attn(
                    self.ln1(x),
                    padding_mask=padding_mask,
                    return_state=True,
                )
            else:
                a = self.attn(
                    self.ln1(x),
                    padding_mask=padding_mask,
                )
                state = None

            x = x + a
            x = x + self.ffn(self.ln2(x))

            return (x, state) if return_state else x

    @torch.no_grad()
    def step(self, x_t, state):
        S, z = state
        a, S, z = self.attn.step(self.ln1(x_t), S, z)
        x = x_t + a
        x = x + self.ffn(self.ln2(x))
        return x, (S, z)

def sinusoidal_positional_encoding(max_len, d_model):
    pos = torch.arange(max_len, dtype=torch.float32)[:, None]
    i = torch.arange(d_model // 2, dtype=torch.float32)[None, :]
    ang = pos / (10000 ** (2 * i / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(ang), torch.cos(ang)
    return pe


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                                 persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        with torch.no_grad():
                nn.init.uniform_(
                    self.tok_emb.weight,
                    -1 / math.sqrt(cfg.d_model),
                    1 / math.sqrt(cfg.d_model)
                )

    def forward(self, idx, padding_mask=None):
        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":
            T = idx.shape[1]
            x = x + self.pos_emb[:T].unsqueeze(0)

        for block in self.blocks:
            x = block(x, padding_mask=padding_mask)

        return self.head(self.ln_f(x))

    # ---------- inference ----------
    @torch.no_grad()
    def prefill(self, idx):
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            T = idx.shape[1]
            x = x + self.pos_emb[start:start + T]
        states = []
        for block in self.blocks:
            x, st = block(x, return_state=True)
            states.append(st)
        return self.head(self.ln_f(x[:, -1:])), states          # logits of last token

    @torch.no_grad()
    def step(self, tok, pos, states):
        x = self._embed(tok, start=pos)                          # tok: (B,1)
        new = []
        for block, st in zip(self.blocks, states):
            x, st = block.step(x, st)
            new.append(st)
        return self.head(self.ln_f(x)), new

In [ ]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE, d_model=MODEL_DIM, n_layers=MODEL_LAYERS,
    n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN, dropout=0.1,
    pos_emb_type="sinusoidal", feat_dim=FEAT_DIM, chunk=ATTN_CHUNK
)
backbone = DecoderLM(cfg).to(device)
assert Path(KERNEL_CHECKPOINT_PATH).exists(), f"Kernel checkpoint missing: {KERNEL_CHECKPOINT_PATH}"
ckpt = torch.load(KERNEL_CHECKPOINT_PATH, map_location='cpu', weights_only=False)  # only trusted checkpoint
state = ckpt.get('model_state_dict',ckpt) if isinstance(ckpt,dict) else ckpt
state = {k.removeprefix('module.'):v for k,v in state.items()}
backbone.load_state_dict(state, strict=True)
print('Loaded kernel checkpoint STRICTLY:',KERNEL_CHECKPOINT_PATH)

In [ ]:
class LoraLinear(nn.Module):
    def __init__(self, layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()

        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout)

        # Freeze original layer
        for p in self.layer.parameters():
            p.requires_grad = False

        self.in_features = layer.in_features
        self.out_features = layer.out_features

        # A: [rank, in_features]
        self.A = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        # B: [out_features, rank]
        self.B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5
        )

    def forward(self, x):

        # Original path: W0 x
        W0x = self.layer(x)

        # LoRA path: BAx
        Ax = F.linear(self.dropout(x), self.A)
        BAx = F.linear( Ax, self.B)

        return W0x + (self.alpha / math.sqrt(self.rank)) * BAx

In [ ]:
class KernelLoraRRClassifier(nn.Module):
    def __init__(self, pretrained_model, num_classes, rank=16, alpha=16,
                 lora_dropout=0.05, dropout=0.2,
                 projections=("q_proj","k_proj","v_proj","out_proj")):
        super().__init__()
        self.model = pretrained_model
        for p in self.model.parameters():
            p.requires_grad_(False)
        inserted = []
        for block_idx, block in enumerate(self.model.blocks):
            attn = block.attn
            for name in projections:
                layer = getattr(attn, name, None)
                if not isinstance(layer, nn.Linear):
                    raise TypeError(f'Expected Linear at kernel block {block_idx}.{name}; got {type(layer)}')
                setattr(attn, name, LoraLinear(layer, rank=rank, alpha=alpha, dropout=lora_dropout))
                inserted.append(f'blocks.{block_idx}.attn.{name}')
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.model.cfg.d_model, num_classes)
        print('LoRA replacements:',len(inserted))
        assert len(inserted) == len(self.model.blocks)*len(projections)

    def forward(self, input_ids, padding_mask, target_mask, rel_pos=None):
        B,T = input_ids.shape
        if T > self.model.cfg.max_seq_len: raise ValueError(f'Input length {T} > {self.model.cfg.max_seq_len}')
        h = self.model.tok_emb(input_ids)
        h = h + self.model.pos_emb[:T].to(h.dtype).unsqueeze(0)
        for block in self.model.blocks:
            h = block(h, padding_mask=padding_mask)
        h = self.model.ln_f(h)
        selected = target_mask.bool() & (~padding_mask.bool())
        counts = selected.sum(dim=1,keepdim=True)
        if (counts == 0).any(): raise ValueError('Empty target sentence after context assembly')
        pooled = (h * selected.unsqueeze(-1)).sum(dim=1) / counts.clamp_min(1)
        return self.head(self.dropout(pooled))

model = KernelLoraRRClassifier(backbone, num_classes=NUM_CLASSES,
    rank=LORA_RANK, alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    projections=LORA_PROJECTIONS).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable {trainable:,} / {total:,} = {trainable/total:.2%}')
assert not model.model.tok_emb.weight.requires_grad

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


# ============================================================
# 1. CONFIGURATION
# ============================================================

PAD_ID = 0
SEP_ID = 3

NUM_CLASSES = 13
MAX_SEQ_LEN = 1024
MAX_TARGET_LEN = 512
MAX_CONTEXT_SENT_LEN = 128
NUM_NEIGHBORS = 5

ID2LABEL = {
    0: "Fact",
    1: "Issue",
    2: "ArgumentPetitioner",
    3: "ArgumentRespondent",
    4: "Statute",
    5: "Dissent",
    6: "PrecedentReliedUpon",
    7: "PrecedentNotReliedUpon",
    8: "PrecedentOverruled",
    9: "RulingByLowerCourt",
    10: "RatioOfTheDecision",
    11: "RulingByPresentCourt",
    12: "None",
}


# ============================================================
# 2. BUILD CONTEXT AROUND EACH TARGET SENTENCE
# ============================================================

def build_context(
    prev_sents,
    target,
    next_sents,
    max_len=MAX_SEQ_LEN,
    max_target=MAX_TARGET_LEN,
    max_ctx=MAX_CONTEXT_SENT_LEN,
    sep_id=SEP_ID,
):
    """
    prev_sents and next_sents are ordered nearest-first.
    Keep the entire truncated target sentence and use remaining
    space for neighbouring sentences.
    """

    target = target[:max_target]

    if len(target) == 0:
        raise ValueError("Encountered an empty target token sequence.")

    # Reserve space for the target first.
    budget = max_len - len(target)

    left_parts = []
    right_parts = []

    li = ri = 0
    take_left = True

    while budget > 0 and (
        li < len(prev_sents) or ri < len(next_sents)
    ):
        added = False

        if take_left and li < len(prev_sents):
            sentence = prev_sents[li]
            li += 1

            # Keep the end of the preceding sentence.
            available = min(max_ctx, max(0, budget - 1))
            piece = sentence[-available:] if available else []

            if piece:
                left_parts.append(piece + [sep_id])
                budget -= len(piece) + 1
                added = True

        elif not take_left and ri < len(next_sents):
            sentence = next_sents[ri]
            ri += 1

            # Keep the beginning of the following sentence.
            available = min(max_ctx, max(0, budget - 1))
            piece = sentence[:available]

            if piece:
                right_parts.append([sep_id] + piece)
                budget -= len(piece) + 1
                added = True

        # If one side has run out, try the other side next.
        take_left = not take_left

        if not added and li >= len(prev_sents) and ri >= len(next_sents):
            break

    left_ids = [
        token
        for part in reversed(left_parts)
        for token in part
    ]
    right_ids = [
        token
        for part in right_parts
        for token in part
    ]

    input_ids = left_ids + target + right_ids

    # 1 for target tokens; 0 for context tokens.
    target_mask = (
        [0] * len(left_ids)
        + [1] * len(target)
        + [0] * len(right_ids)
    )

    assert len(input_ids) <= max_len
    assert len(input_ids) == len(target_mask)

    return input_ids, target_mask


# ============================================================
# 3. DATASET
# ============================================================

class RRContextDataset(Dataset):
    def __init__(
        self,
        ids,
        labels,
        doc_ids,
        k=NUM_NEIGHBORS,
    ):
        assert len(ids) == len(labels) == len(doc_ids)

        self.ids = ids
        self.labels = labels
        self.doc_ids = doc_ids
        self.k = k

        # Labels must be integer IDs in [0, NUM_CLASSES).
        assert all(0 <= int(y) < NUM_CLASSES for y in labels)

        # Build document boundaries.
        n = len(ids)
        self.start = [0] * n
        self.end = [0] * n

        runs = 1 + sum(
            doc_ids[i] != doc_ids[i - 1]
            for i in range(1, n)
        ) if n else 0

        assert runs == len(set(doc_ids)), (
            "Document IDs must be contiguous. "
            "Keep sentences in document order and do not "
            "interleave sentences from different documents."
        )

        i = 0
        while i < n:
            j = i

            while (
                j + 1 < n
                and doc_ids[j + 1] == doc_ids[i]
            ):
                j += 1

            for t in range(i, j + 1):
                self.start[t] = i
                self.end[t] = j

            i = j + 1

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        start = self.start[i]
        end = self.end[i]

        # Nearest previous sentence first.
        prev_sents = [
            self.ids[j]
            for j in range(
                i - 1,
                max(start, i - self.k) - 1,
                -1,
            )
        ]

        # Nearest following sentence first.
        next_sents = [
            self.ids[j]
            for j in range(
                i + 1,
                min(end, i + self.k) + 1,
            )
        ]

        input_ids, target_mask = build_context(
            prev_sents,
            self.ids[i],
            next_sents,
        )

        # Relative position within the document.
        rel_pos = (i - start) / max(end - start, 1)

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_mask, dtype=torch.bool),
            torch.tensor(rel_pos, dtype=torch.float32),
            torch.tensor(self.labels[i], dtype=torch.long),
        )


# ============================================================
# 4. COLLATE FUNCTION
# ============================================================

def rr_collate_fn(batch):
    input_ids, target_masks, rel_positions, labels = zip(*batch)

    lengths = torch.tensor([len(x) for x in input_ids])

    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=PAD_ID,
    )

    target_masks = pad_sequence(
        target_masks,
        batch_first=True,
        padding_value=False,
    )

    positions = torch.arange(input_ids.size(1))

    # True means padding for this model's mask convention.
    padding_mask = positions.unsqueeze(0) >= lengths.unsqueeze(1)

    return {
        "input_ids": input_ids,
        "padding_mask": padding_mask,
        "target_mask": target_masks,
        "rel_pos": torch.stack(rel_positions),
        "labels": torch.stack(labels),
    }


# ============================================================
# 5. RR CLASSIFIER
# ============================================================

class RRClassifier(nn.Module):
    def __init__(
        self,
        base_model,
        d_model,
        num_classes=NUM_CLASSES,
        dropout=0.2,
        use_pos=False,
    ):
        super().__init__()

        self.model = base_model
        self.use_pos = use_pos
        self.drop = nn.Dropout(dropout)

        self.head = nn.Linear(
            d_model + int(use_pos),
            num_classes,
        )

    def get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        x = self.model.tok_emb(idx)
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T]
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError("padding_mask must have shape (B, T)")
            padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
        for block in self.model.blocks:
            x, _ = block(x, padding_mask=padding_mask)
        return self.model.ln_f(x)

    def forward(
        self,
        input_ids,
        padding_mask,
        target_mask,
        rel_pos=None,
    ):
        hidden = self.get_hidden_states(
            input_ids,
            padding_mask,
        )  # [B, T, d_model]

        # Pool only target-sentence tokens.
        valid_target = (
            target_mask.bool()
            & ~padding_mask.bool()
        ).unsqueeze(-1).to(hidden.dtype)

        target_count = valid_target.sum(dim=1)

        if torch.any(target_count == 0):
            raise ValueError(
                "A sample contains no valid target tokens."
            )

        pooled = (
            (hidden * valid_target).sum(dim=1)
            / target_count.clamp(min=1)
        )  # [B, d_model]

        if self.use_pos:
            if rel_pos is None:
                raise ValueError(
                    "rel_pos is required when use_pos=True."
                )

            pooled = torch.cat(
                [
                    pooled,
                    rel_pos.unsqueeze(-1).to(pooled.dtype),
                ],
                dim=-1,
            )

        logits = self.head(self.drop(pooled))

        return logits  # [B, 13]





# ============================================================
# 7. EVALUATION
# ============================================================

In [ ]:
def build_rr_data(data, tokenizer, label2id, max_seq_len=1024):
    train_ids = []
    train_labels = []
    train_doc_ids = []

    documents = data.values() if isinstance(data, dict) else data

    for doc in documents:
        doc_id = str(doc["id"])
        sentences = doc["text"]
        labels = doc["labels"]

        assert len(sentences) == len(labels), (
            f"Sentence-label mismatch in document {doc_id}"
        )

        for sentence, label in zip(sentences, labels):
            token_ids = converter.to_tokens([sentence])[0]
            token_ids = token_ids[:max_seq_len]

            train_ids.append(token_ids)
            train_labels.append(label2id[label])
            train_doc_ids.append(doc_id)

    return train_ids, train_labels, train_doc_ids

In [ ]:
def split_to_records(split):
    ids, targets, docs = [], [], []
    for domain in ('cl','it'):
        a,b,c = build_rr_data(raw[f'{domain}_{split}'], converter, labels_data, max_seq_len=MAX_SEQ_LEN)
        # Prefix doc IDs by domain so documents from CL and IT never share a boundary.
        docs.extend([f'{domain}:{x}' for x in c]); ids.extend(a); targets.extend(b)
    assert len(ids)==len(targets)==len(docs)
    return ids,targets,docs
train_ids,train_labels,train_doc_ids = split_to_records('train')
val_ids,val_labels,val_doc_ids = split_to_records('val')
train_ds = RRContextDataset(train_ids,train_labels,train_doc_ids,k=NUM_NEIGHBORS)
val_ds = RRContextDataset(val_ids,val_labels,val_doc_ids,k=NUM_NEIGHBORS)
print('Train sentences:',len(train_ds),'Val sentences:',len(val_ds))

In [ ]:
# Check data interfaces and trainability without changing pretrained weights permanently.
probe = rr_collate_fn([train_ds[0]])
probe = {key: val.to(device) for key,val in probe.items()}
model.train()
logits = model(probe['input_ids'],probe['padding_mask'],probe['target_mask'],probe['rel_pos'])
assert logits.shape == (1,NUM_CLASSES)
loss = F.cross_entropy(logits,probe['labels']); loss.backward()
assert model.head.weight.grad is not None
assert model.model.blocks[0].attn.q_proj.B.grad is not None
assert model.model.tok_emb.weight.grad is None
model.zero_grad(set_to_none=True)
print('Sanity check PASSED; output shape:',tuple(logits.shape))

In [ ]:
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                          collate_fn=rr_collate_fn,num_workers=0)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,
                        collate_fn=rr_collate_fn,num_workers=0)
counts = np.bincount(np.asarray(train_labels),minlength=NUM_CLASSES).astype(np.float32)
w = np.zeros(NUM_CLASSES,dtype=np.float32)
present=counts>0
w[present] = counts[present] ** (-0.25)
w[present] /= w[present].mean()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w,device=device), label_smoothing=0.05)
optimizer = torch.optim.AdamW([
    {'params':[p for p in model.model.parameters() if p.requires_grad],'lr':5e-4},
    {'params':list(model.head.parameters()),'lr':1e-3},
],betas=(0.9,0.98),weight_decay=0.01)

@torch.no_grad()
def validate(model, loader):
    model.eval(); y_true=[]; y_pred=[]
    for batch in loader:
        batch={k:v.to(device) for k,v in batch.items()}
        out=model(batch['input_ids'],batch['padding_mask'],batch['target_mask'],batch['rel_pos'])
        y_true.extend(batch['labels'].cpu().tolist());y_pred.extend(out.argmax(-1).cpu().tolist())
    return {
        'macro_f1': f1_score(y_true,y_pred,average='macro',labels=list(range(NUM_CLASSES)),zero_division=0),
        'macro_precision': precision_score(y_true,y_pred,average='macro',labels=list(range(NUM_CLASSES)),zero_division=0),
        'macro_recall': recall_score(y_true,y_pred,average='macro',labels=list(range(NUM_CLASSES)),zero_division=0),
        'micro_f1': f1_score(y_true,y_pred,average='micro',zero_division=0),
    }

best=-1.;bad=0;history=[]; best_path=OUTPUT_DIR/'best_kernel_lora_rr.pt'
for epoch in range(1,EPOCHS+1):
    model.train(); total=0.;seen=0
    for batch in train_loader:
        batch={k:v.to(device) for k,v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        logits=model(batch['input_ids'],batch['padding_mask'],batch['target_mask'],batch['rel_pos'])
        loss=criterion(logits,batch['labels'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
        optimizer.step()
        n=len(batch['labels']);total+=loss.item()*n;seen+=n
    metrics=validate(model,val_loader)
    metrics.update(epoch=epoch,train_loss=total/max(seen,1))
    history.append(metrics)
    print(metrics)
    if metrics['macro_f1']>best:
        best=metrics['macro_f1'];bad=0
        torch.save({'model_state_dict':model.state_dict(),
                    'epoch':epoch,'val_metrics':metrics,
                    'configuration':{'lora_projections':LORA_PROJECTIONS,
                                     'rank':LORA_RANK,'alpha':LORA_ALPHA}},best_path)
    else:
        bad+=1
        if bad>=PATIENCE:
            print('Early stopping');break

print('Best local validation Macro-F1:',best)
with open(OUTPUT_DIR/'training_history.json','w') as f:json.dump(history,f,indent=2)

In [ ]:
ck = torch.load(best_path,map_location='cpu',weights_only=False)
model.load_state_dict(ck['model_state_dict'],strict=True)
model.to(device);model.eval()
assert not(set(map(str,raw['cl_test'])) & set(map(str,raw['it_test']))), 'CL/IT test IDs collide'

In [ ]:
class RRTestDataset(Dataset):
    def __init__(self, ids, doc_ids, k=NUM_NEIGHBORS):
        assert len(ids) == len(doc_ids)

        self.ids = ids
        self.doc_ids = [str(x) for x in doc_ids]
        self.k = k

        n = len(ids)
        self.start = [0] * n
        self.end = [0] * n

        runs = (
            1 + sum(
                self.doc_ids[i] != self.doc_ids[i - 1]
                for i in range(1, n)
            )
            if n else 0
        )

        assert runs == len(set(self.doc_ids)), (
            "Document IDs must be contiguous."
        )

        i = 0
        while i < n:
            j = i

            while (
                j + 1 < n
                and self.doc_ids[j + 1] == self.doc_ids[i]
            ):
                j += 1

            for t in range(i, j + 1):
                self.start[t] = i
                self.end[t] = j

            i = j + 1

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        start, end = self.start[i], self.end[i]

        prev_sents = [
            self.ids[j]
            for j in range(
                i - 1,
                max(start, i - self.k) - 1,
                -1,
            )
        ]

        next_sents = [
            self.ids[j]
            for j in range(
                i + 1,
                min(end, i + self.k) + 1,
            )
        ]

        input_ids, target_mask = build_context(
            prev_sents,
            self.ids[i],
            next_sents,
        )

        rel_pos = (i - start) / max(end - start, 1)

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_mask, dtype=torch.bool),
            torch.tensor(rel_pos, dtype=torch.float32),
        )
def rr_test_collate_fn(batch):
    input_ids, target_masks, rel_positions = zip(*batch)

    lengths = torch.tensor([len(x) for x in input_ids])

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=PAD_ID,
    )

    target_masks = torch.nn.utils.rnn.pad_sequence(
        target_masks,
        batch_first=True,
        padding_value=False,
    )

    positions = torch.arange(input_ids.size(1))
    padding_mask = positions.unsqueeze(0) >= lengths.unsqueeze(1)

    return {
        "input_ids": input_ids,
        "padding_mask": padding_mask,
        "target_mask": target_masks,
        "rel_pos": torch.stack(rel_positions),
    }

def build_rr_test_data(data, tokenizer, max_seq_len=MAX_SEQ_LEN):
    test_ids = []
    test_doc_ids = []

    for doc_id, sentences in data.items():
        doc_id = str(doc_id)

        for sentence in sentences:
            token_ids = tokenizer.to_tokens([sentence])[0]
            token_ids = token_ids[:max_seq_len]

            test_ids.append(token_ids)
            test_doc_ids.append(doc_id)

    return test_ids, test_doc_ids

In [ ]:
test_data = {**raw['cl_test'],**raw['it_test']}
test_ids,test_doc_ids=build_rr_test_data(test_data,converter,max_seq_len=MAX_SEQ_LEN)
test_ds = RRTestDataset(test_ids,test_doc_ids,k=NUM_NEIGHBORS)
assert len(test_ds)==sum(len(v) for v in test_data.values())

@torch.no_grad()
def make_predictions():
    loader=DataLoader(test_ds,batch_size=BATCH_SIZE,collate_fn=rr_test_collate_fn,num_workers=0)
    preds={}
    model.eval()
    for batch in loader:
        batch={k:v.to(device) for k,v in batch.items()}
        scores=model(batch['input_ids'],batch['padding_mask'],batch['target_mask'],batch['rel_pos'])
        label_ids=scores.argmax(-1).cpu().tolist()
        offset=sum(len(v) for v in preds.values())
        for doc_id,pred_id in zip(test_ds.doc_ids[offset:offset+len(label_ids)],label_ids):
            preds.setdefault(doc_id,[]).append(id2label[pred_id])
    assert len(preds)==len(test_data)
    for doc_id,sents in test_data.items():
        assert len(preds[str(doc_id)])==len(sents), f'Mismatch: {doc_id}'
    return preds

predictions = make_predictions()
pred_file=OUTPUT_DIR/'prediction.json'
with open(pred_file,'w',encoding='utf-8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)
zip_file=OUTPUT_DIR/'submission.zip'
with zipfile.ZipFile(zip_file,'w',compression=zipfile.ZIP_DEFLATED) as f:
    f.write(pred_file,arcname='prediction.json')
print('Saved',zip_file,'documents',len(predictions))